## Tutorial Amazon AgentCore Bedrock Code Interpreter

Este tutorial demonstra como usar o AgentCore Bedrock Code Interpreter para:
1. Configurar um ambiente sandbox
2. Carregar e analisar dados
3. Executar código em um ambiente sandbox
4. Processar e recuperar resultados

## Pré-requisitos
- Conta AWS com acesso ao Bedrock AgentCore Code Interpreter
- Você deve ter as permissões IAM necessárias para criar e gerenciar recursos do code interpreter
- Pacotes Python necessários instalados (incluindo boto3 & bedrock-agentcore)
- Arquivo de dados de exemplo (data.csv)
- Script de análise (stats.py)


## Seu role de execução IAM deve ter a seguinte política IAM anexada

~~~ {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:CreateCodeInterpreter",
                "bedrock-agentcore:StartCodeInterpreterSession",
                "bedrock-agentcore:InvokeCodeInterpreter",
                "bedrock-agentcore:StopCodeInterpreterSession",
                "bedrock-agentcore:DeleteCodeInterpreter",
                "bedrock-agentcore:ListCodeInterpreters",
                "bedrock-agentcore:GetCodeInterpreter"
            ],
            "Resource": "*"
        },
        {
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents"
            ],
            "Resource": "arn:aws:logs:*:*:log-group:/aws/bedrock-agentcore/code-interpreter*"
        }
    ]
}



## Como funciona

O sandbox de execução de código permite que agentes processem consultas de usuários com segurança, criando um ambiente isolado com um interpretador de código, shell e sistema de arquivos. Após um Large Language Model auxiliar na seleção de ferramentas, o código é executado dentro desta sessão, antes de ser retornado ao usuário ou agente para síntese.

![architecture local](code-interpreter.png)

## 1. Configurando o Ambiente

Primeiro, vamos importar as bibliotecas necessárias e inicializar nosso cliente Code Interpreter.

In [ ]:
!pip install --upgrade -r requirements.txt

In [21]:
from bedrock_agentcore.tools.code_interpreter_client import CodeInterpreter
import json
from typing import Dict, Any, List

# Inicializar o Code Interpreter com sua região AWS
code_client = CodeInterpreter('us-west-2')
code_client.start()

'01K00Z3F8WZ9KBBW4QGRJCVBHH'

## 2. Lendo Arquivos Locais

Agora vamos ler o conteúdo do nosso arquivo de dados de exemplo e script de análise.

In [22]:
def read_file(file_path: str) -> str:
    """Função auxiliar para ler o conteúdo do arquivo com tratamento de erros"""
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            return file.read()
    except FileNotFoundError:
        print(f"Erro: O arquivo '{file_path}' não foi encontrado.")
        return ""
    except Exception as e:
        print(f"Ocorreu um erro: {e}")
        return ""

# Ler ambos os arquivos
data_file_content = read_file("samples/data.csv")
code_file_content = read_file("samples/stats.py")

## 3. Preparando Arquivos para o Ambiente Sandbox

Vamos criar uma estrutura que define os arquivos que queremos criar no ambiente sandbox.

In [23]:
files_to_create = [
    {
        "path": "data.csv",
        "text": data_file_content
    },
    {
        "path": "stats.py",
        "text": code_file_content
    }
]

## 4. Criando Função Auxiliar para Invocação de Ferramentas

Esta função auxiliar facilitará a chamada de ferramentas do sandbox e o tratamento de suas respostas. Dentro de uma sessão ativa, você pode executar código em linguagens suportadas (Python, JavaScript), acessar bibliotecas com base na configuração de dependências, gerar visualizações e manter o estado entre execuções.

In [24]:
def call_tool(tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    """Função auxiliar para invocar ferramentas do sandbox
    
    Args:
        tool_name (str): Nome da ferramenta a invocar
        arguments (Dict[str, Any]): Argumentos a passar para a ferramenta
        
    Returns:
        Dict[str, Any]: Resultado formatado em JSON
    """
    response = code_client.invoke(tool_name, arguments)
    for event in response["stream"]:
        return json.dumps(event["result"])

## 5. Escrevendo Arquivos no Sandbox

Agora vamos escrever nossos arquivos no ambiente sandbox e verificar se foram criados com sucesso.

In [25]:
# Escrever arquivos no sandbox
writing_files = call_tool("writeFiles", {"content": files_to_create})
print("Resultado da escrita de arquivos:")
print(writing_files)

# Verificar se os arquivos foram criados
listing_files = call_tool("listFiles", {"path": ""})
print("\nArquivos no sandbox:")
print(listing_files)

Resultado da escrita de arquivos:
{"content": [{"type": "text", "text": "Successfully wrote all 2 files"}], "isError": false}

Arquivos no sandbox:
{"content": [{"type": "resource_link", "uri": "file:///log", "name": "log", "description": "Directory"}, {"type": "resource_link", "mimeType": "text/csv", "uri": "file:///data.csv", "name": "data.csv", "description": "File"}, {"type": "resource_link", "mimeType": "text/x-python", "uri": "file:///stats.py", "name": "stats.py", "description": "File"}, {"type": "resource_link", "uri": "file:///.ipython", "name": ".ipython", "description": "Directory"}], "isError": false}


## 6. Executando a Análise

Agora vamos executar nosso script de análise no ambiente sandbox e processar os resultados.

In [26]:
import pprint

# Executar o script de análise
code_execute_result = call_tool("executeCode", {
    "code": files_to_create[1]['text'],
    "language": "python",
    "clearContext": True
})

# Analisar e exibir resultados
analysis_results = json.loads(code_execute_result)
print("Resultados completos da análise:")
pprint.pprint(analysis_results)

print("\nSaída padrão da análise:")
print(analysis_results['structuredContent']['stdout'])

Resultados completos da análise:
{'content': [{'text': 'Name   Place  Animal   Thing\n'
                      'count       299130  299130  299130  299130\n'
                      'unique        1722      55      50      51\n'
                      'top     Lisa White  Prague    Goat  Pencil\n'
                      'freq           222    5587    6141    6058',
              'type': 'text'}],
 'isError': False,
 'structuredContent': {'executionTime': 0.7111423015594482,
                       'exitCode': 0,
                       'stderr': '',
                       'stdout': 'Name   Place  Animal   Thing\n'
                                 'count       299130  299130  299130  299130\n'
                                 'unique        1722      55      50      51\n'
                                 'top     Lisa White  Prague    Goat  Pencil\n'
                                 'freq           222    5587    6141    6058'}}

Saída padrão da análise:
Name   Place  Animal   Thing
count     

## 7. Limpeza

Por fim, vamos fazer a limpeza parando a sessão do Code Interpreter. Ao terminar de usar uma sessão, ela deve ser interrompida para liberar recursos e evitar cobranças desnecessárias.

In [27]:
# Parar a sessão do Code Interpreter
code_client.stop()
print("Sessão do Code Interpreter parada com sucesso!")

Sessão do Code Interpreter parada com sucesso!


## Conclusão

Neste tutorial, aprendemos como:
- Inicializar uma sessão do Code Interpreter
- Ler e preparar arquivos para análise
- Executar código em um ambiente sandbox
- Processar e exibir resultados
- Limpar recursos